In [ ]:
import gymnasium as gym
from gymnasium import spaces
import math
import numpy as np
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from treys import Card, Deck, Evaluator

evalu = Evaluator()

# -----------------------------------------------------------
# Monte Carlo Equity Calculator
# -----------------------------------------------------------

def simulate_preflop(h1, h2):

    your_hand = [Card.new(h1), Card.new(h2)]

    count = 0


    N = 2500  

    for i in range(N):
        deck = Deck()

        # remove hero hand from deck
        deck.cards.remove(your_hand[0])
        deck.cards.remove(your_hand[1])

        villain_hand = deck.draw(2)
        board = deck.draw(5)

        hero_score = evalu.evaluate(board, your_hand)
        villain_score = evalu.evaluate(board, villain_hand)


        if villain_score > hero_score:
            count += 1
        else:
            pass

    win_rate = count / N


    return win_rate

def simulate_preturn(h1, h2, b1, b2, b3):

    your_hand = [Card.new(h1), Card.new(h2)]
    flop = [Card.new(b1), Card.new(b2), Card.new(b3)]

    count = 0

    N = 2500

    for i in range(N):
        deck = Deck()

        # remove hero cards and flop cards
        deck.cards.remove(your_hand[0])
        deck.cards.remove(your_hand[1])
        for c in flop:
            deck.cards.remove(c)

        villain_hand = deck.draw(2)
        turn_river = deck.draw(2)
        board = flop + turn_river

        hero_score = evalu.evaluate(board, your_hand)
        villain_score = evalu.evaluate(board, villain_hand)


        if villain_score > hero_score:
            count += 1
        else:
            pass

    win_rate = count / N


    return win_rate

def simulate_preriver(h1, h2, b1, b2, b3, b4):

    your_hand = [Card.new(h1), Card.new(h2)]
    flop_turn = [
        Card.new(b1), 
        Card.new(b2), 
        Card.new(b3), 
        Card.new(b4)
    ]

    count = 0

    N = 2500

    for i in range(N):
        deck = Deck()

        deck.cards.remove(your_hand[0])
        deck.cards.remove(your_hand[1])

        for c in flop_turn:
            deck.cards.remove(c)

        villain_hand = deck.draw(2)
        river = deck.draw(1)
        board = flop_turn + river

        hero_score = evalu.evaluate(board, your_hand)
        villain_score = evalu.evaluate(board, villain_hand)


        if villain_score > hero_score:
            count += 1
        else:
            pass

    win_rate = count / N

    return win_rate

def simulate_prereveal(h1, h2, b1, b2, b3, b4, b5):

    your_hand = [Card.new(h1), Card.new(h2)]
    board = [
        Card.new(b1),
        Card.new(b2),
        Card.new(b3),
        Card.new(b4),
        Card.new(b5)
    ]

    hero_score = evalu.evaluate(board, your_hand)

    count = 0

    N = 2500

    for i in range(N):
        deck = Deck()
        villain_hand = deck.draw(2)

        villain_score = evalu.evaluate(board, villain_hand)

        if villain_score > hero_score:
            count += 1
        else:
            pass

    win_rate = count / N

    return win_rate


# -----------------------------------------------------------
# Environment 
# -----------------------------------------------------------

class Poker:
    
    def __init__(self):
    """
    Notes for __init__ function:
    Pot, hero stack, effective stack, are normalized by using rate-based adjustment - dividing by starting stack. 
    While starting stack is normalized by using min-max rescaling - x - x_min/ x_max - x_min.
    """
        super().__init__()

        self.observation_space = spaces.Box(
            low = np.array([
                0.0, # win rate
                0.0, # starting stack (starting stacks are equal for both opp and hero)
                0.0, # pot
                0.0, # hero stack
                0.0, # effective stack
                0.0, # position
                0.0, # pot odds
                0.0, # opp agg
                0.0, # street
                0.0  # opp last action
            ], dtype = np.float32),
            high = np.array([
                1.0,
                1.0, 
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                5.0,
                1.0,
                6.0
            ], dtype = np.float32),
            dtype = np.float32
        )

        self.action_space = spaces.Discrete(7) # 0 -> fold, 1 -> check/call, 2 -> bet 33% of pot, 3 -> bet 67% of pot, 4 -> bet 100% of pot, 5 -> bet 150% of pot, 5 -> all-in
        
        self.button = None # 0 for small blind, 1 for big blind
        self.small_blind = None # will be set to 0.02x of stack
        self.big_blind = None # big blind is 2x small blind
        self.stack = None # will be randomly set within a normal range

        self.deck = None
        self.street = None # 0 --> preflop, 1 --> after flop, 2 --> after turn, 3 --> after after river
        self.board = None
        self.h1 = None
        self.h2 = None

        self.opp_action = None # will mirror action space
        self.opp_raises = None
        self.opp_calls = None
       
        self.bet_amount = None # bet amount (will be on big blind for every street except 0th - preflop)
        self.bet_history = None # will be a list of current bet and bet on the last iteration, can get call/raise amounts, and a chekc if betting is finished
       
        self.chips_to_pot = None # chips betting is symmetric so we only need one, must createa  funciton that checks if other person calls though
       
        self.win_rate = None
    

    def _get_obs(self):
        call_amount = self.bet_history[1] - self.bet_history[0]
        
        if call_amount == 0:
            pot_odds = 0
        else:
            pot_odds = call_amount / (self.pot + call_amount)

        opp_agg = min(self.opp_raises / max(self.opp_calls, 1), 5)
        return np.array([
            self.win_rate, 
            (self.stack - 100) / 400
            self.pot / (2 * self.starting_stack), # normalized pot
            self.hero_stack / self.starting_stack, # normalized hero stack
            min(self.hero_stack, self.opp_stack) / self.starting_stack, # normalized effective stack
            self.position,
            pot_odds, # pot odds (doesn't inlcude opponent last bet as it is already accounted for in the pot)
            opp_agg, 
            self.street / 3,
            self.opp_last_action
        ], dtype = np.float32)

    
"""
Notes for this class:
I did not add any info function as I don't find it useful in my debugging in this modern age with AI, and I could use the extra processing power.
"""

    def reset(self):
        self.button = random.randint(0,1)
        self.stack = random.randint(100, 500)
        self.small_blind = self.stack * 0.02
        self.big_blind = 2 * self.small_blind
        self.street = 0

        self.opp_raises = 0
        self.opp_calls = 0

        self.bet_amount = 0
        self.bet_history = deque(maxlen=2)
       
        self.chips_to_pot = 0
        self.win_rate = 0


    def compute_win_rate(self):
        """ Notes for compute_win_rate function:
        Don't know if I need return statement since the variable being updated is global (within the class)
        """

        if self.street == 0:
            self.win_rate = simulate_preflop(self.h1, self.h2)
            return self.win_rate
            
        elif self.street == 1:
            self.win_rate = simulate_preturn(self.h1, self.h2, self.board[0], self.board[1], self.board[2])
            return self.win_rate
            
        elif self.street == 2:
            self.win_rate = simulate_preriver(self.h1, self.h2, self.board[0], self.board[1], self.board[2], self.board[3])
            return self.win_rate
        else:
            self.win_rate = simulate_prereveal(self.h1, self.h2, self.board[0], self.board[1], self.board[2], self.board[3], self.board[4])
            return self.win_rate


    def first_to_act(self):
"""
Notes for first_to_act function:
Can we negate the while loops entirely?
This function will return who is first to act at the beginning of each street
This function should be called after the incremation of the street in step()
"""
        if self.button == 0:
            if self.street == 0:
                return "hero"
            else:
                while self.street in (1,2,3):
                    return "opp"
        else:
        # opponent has button
            if self.street = 0:
                return "opp"
            else:
                while self.street in (1,2,3):
                    return "hero"
   
    self.opp_action = random.randint(0,6)


